# Predictive Maintenance: EDA & How the Model Is Built

> ⚠️ Synthetic data for educational purposes only.

This notebook walks through the workflow behind the dashboard:

1. **Explore the raw data**: sensors, equipment types, missing values, spikes
2. **Explore failures**: how often, which modes, what the signals look like *before* a failure
3. **Feature engineering**: hourly readings turned into daily features with no data leakage
4. **Modeling**: time-based split, a naive baseline, a Random Forest, metrics, thresholds, feature importance

It reuses the project code in `src/`, so the notebook and the app always agree.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Find the project root (the folder that contains src/) so `import src` works
ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.generate_data import EQUIPMENT_TYPES, SENSORS, generate, load_raw
from src.features import FEATURE_COLUMNS, LABEL, build_features
from src.train import time_split

DATA_DIR = ROOT / "data"
if not (DATA_DIR / "sensor_readings.csv").exists():
    generate(out_dir=DATA_DIR)  # same seed (42) -> same data as the app

pd.set_option("display.max_columns", 50)
print("Project root:", ROOT)

## 1. Raw data overview
Three files: hourly **sensor readings**, **failure events**, and the **maintenance log**.

In [ ]:
readings, failures, maintenance = load_raw(DATA_DIR)

print(f"sensor_readings : {readings.shape[0]:,} rows x {readings.shape[1]} cols")
print(f"failure_events  : {len(failures)} failures")
print(f"maintenance_log : {len(maintenance)} maintenance visits")
print(f"Period          : {readings.timestamp.min()} -> {readings.timestamp.max()}")
print(f"Units           : {readings.equipment_id.nunique()} ({readings.groupby('equipment_type').equipment_id.nunique().to_dict()})")
readings.head()

In [ ]:
# Each type has its own normal range, so a global threshold (e.g. 'vibration > 4') would not work
readings.groupby("equipment_type")[SENSORS].mean().round(2)

In [ ]:
missing = (readings[SENSORS].isna().mean() * 100).round(2).rename("% missing")
print("About 1% missing per sensor. features.py forward-fills these, using only past values.")
missing.to_frame()

## 2. Sensor distributions
Box plots by equipment type. The dots far above each box are **sensor spikes** and **pre-failure degradation**.

In [ ]:
sample = readings.sample(20_000, random_state=42)
for s in ["vibration_mm_s", "temperature_c", "current_amp"]:
    fig = px.box(sample, x="equipment_type", y=s, color="equipment_type",
                 title=f"{s} by equipment type", height=350)
    fig.update_layout(showlegend=False)
    fig.show()

## 3. Failures
How many failures are there, of which type, and how far apart?

In [ ]:
f = failures.merge(readings[["equipment_id", "equipment_type"]].drop_duplicates(), on="equipment_id")
fig = px.histogram(f, x="failure_mode", color="equipment_type", barmode="group",
                   title="Failures by mode and equipment type", height=380)
fig.show()

gaps = f.sort_values("failure_time").groupby("equipment_id").failure_time.diff().dt.days.dropna()
print(f"Failures per unit: min {f.groupby('equipment_id').size().min()}, max {f.groupby('equipment_id').size().max()}")
print(f"Days between failures on the same unit: min {gaps.min():.0f}, median {gaps.median():.0f}")

In [ ]:
unit = failures.equipment_id.iloc[0]
r = readings[readings.equipment_id == unit]
fig = px.line(r, x="timestamp", y=["vibration_mm_s", "temperature_c"], facet_row="variable",
              title=f"{unit}: red = failure, green = maintenance", height=550)
fig.update_yaxes(matches=None)
for t in failures[failures.equipment_id == unit].failure_time:
    fig.add_vline(x=t, line_color="red", line_dash="dash")
for t in maintenance[maintenance.equipment_id == unit].maintenance_date:
    fig.add_vline(x=t, line_color="green", line_dash="dot")
fig.show()

### What happens *before* a failure?
Line up every failure at time 0 and average the signals over the 14 days before it. Each sensor is shown **relative to its type's baseline** (1.0 = normal), so all types can share one chart. **This chart motivates the features we build next.**

In [ ]:
rows = []
for _, ev in failures.iterrows():
    r = readings[(readings.equipment_id == ev.equipment_id)
                 & (readings.timestamp > ev.failure_time - pd.Timedelta(days=14))
                 & (readings.timestamp <= ev.failure_time)].copy()
    base = EQUIPMENT_TYPES[r.equipment_type.iloc[0]]
    r["days_before_failure"] = ((ev.failure_time - r.timestamp).dt.total_seconds() / 86400).round()
    for s in ["vibration_mm_s", "temperature_c", "current_amp", "pressure_bar"]:
        r[s] = r[s] / base[s][0]
    rows.append(r)
aligned = pd.concat(rows)

profile = aligned.groupby("days_before_failure")[["vibration_mm_s", "temperature_c", "current_amp", "pressure_bar"]].mean()
fig = px.line(profile, title="Average signal / baseline in the 14 days before failure",
              labels={"value": "relative to baseline", "days_before_failure": "days before failure"}, height=400)
fig.update_xaxes(autorange="reversed")
fig.show()
print("Vibration and temperature drift up over the last 3-10 days; that is the early warning we want to learn.")

In [ ]:
corr = readings[SENSORS + [LABEL]].corr().round(2)
px.imshow(corr, text_auto=True, color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
          title="Correlation (hourly). Raw values correlate weakly with the label", height=450).show()

## 4. The label and class imbalance
`failure_within_7d = 1` means the unit fails in the next 7 days. Positive days are rare, which matters a lot for how we judge the model.

In [ ]:
rate = readings[LABEL].mean()
print(f"Positive (failure within 7d): {rate:.1%}   Negative: {1 - rate:.1%}")
px.pie(names=["No failure within 7d", "Failure within 7d"],
       values=readings[LABEL].value_counts().sort_index().values,
       title="Label balance (hourly)", height=330).show()

## 5. Feature engineering (`src/features.py`)
One row per unit per day, as of the **end of that day**:
- rolling **mean / std / max** over 24h and 7d for each sensor
- **slope** of vibration and temperature (change per day over the last 3 days): *is it getting worse?*
- **days since last maintenance / failure**
- one-hot **equipment type**

All windows look **backwards only**, so there is no data leakage.

In [ ]:
feats = build_features(readings, failures, maintenance)
print(f"{len(feats):,} daily rows x {len(FEATURE_COLUMNS)} features")
print(f"Daily positive rate: {feats[LABEL].mean():.1%}")
feats[["equipment_id", "date"] + FEATURE_COLUMNS[:6] + ["vibration_slope_3d", "days_since_maintenance", LABEL]].head()

In [ ]:
for col in ["vibration_slope_3d", "temperature_slope_3d", "current_amp_std_24h"]:
    px.box(feats, x=LABEL, y=col, color=LABEL, points=False, height=320,
           title=f"{col}: label 0 vs 1").show()
print("Clear separation means these engineered features carry signal that raw hourly values don't.")

## 6. Time-based train/test split
We **never shuffle**. The model trains on the first 75% of days and is tested on the last 25%, the same way it would be used in real life: learn from the past, predict the future.

In [ ]:
train_df, test_df = time_split(feats, train_frac=0.75)
X_train, y_train = train_df[FEATURE_COLUMNS], train_df[LABEL]
X_test, y_test = test_df[FEATURE_COLUMNS], test_df[LABEL]
for name, d in [("train", train_df), ("test", test_df)]:
    print(f"{name:5}: {d.date.min().date()} -> {d.date.max().date()}  rows={len(d):,}  positive rate={d[LABEL].mean():.1%}")

## 7. Baseline: always predict "no failure"
Always start with a dumb baseline. Here it shows why **accuracy is misleading**.

In [ ]:
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score, precision_recall_curve)

baseline_pred = np.zeros(len(y_test), dtype=int)
print(f"Accuracy: {accuracy_score(y_test, baseline_pred):.1%}  <- looks great...")
print(f"Recall  : {recall_score(y_test, baseline_pred):.1%}   <- ...but it catches ZERO failures")

## 8. Random Forest
- `class_weight="balanced"` makes the rare failure class count as much as the common healthy class
- `max_depth=8` and `min_samples_leaf=3` keep trees from memorising noise
- These are the same settings as `src/train.py`

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=3,
                               class_weight="balanced", random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

proba = model.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)
metrics = pd.Series({
    "precision": precision_score(y_test, pred), "recall": recall_score(y_test, pred),
    "f1": f1_score(y_test, pred), "roc_auc": roc_auc_score(y_test, proba),
    "accuracy": accuracy_score(y_test, pred)}).round(3)
print(metrics.to_string())

cm = confusion_matrix(y_test, pred)
px.imshow(cm, text_auto=True, color_continuous_scale="Blues", height=380,
          x=["Pred: OK", "Pred: Failure"], y=["Actual: OK", "Actual: Failure"],
          title="Confusion matrix (threshold 0.5)").show()
print(f"Missed failures (FN) = {cm[1,0]}, false alarms (FP) = {cm[0,1]}")

### Choosing the threshold: precision vs recall
A missed failure costs far more than an extra inspection. Lowering the threshold catches more failures (**recall up**) at the cost of more false alarms (**precision down**).

In [ ]:
rows = []
for t in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    p = (proba >= t).astype(int)
    rows.append({"threshold": t, "precision": precision_score(y_test, p, zero_division=0),
                 "recall": recall_score(y_test, p), "false_alarms": int(((p == 1) & (y_test == 0)).sum()),
                 "missed_failures": int(((p == 0) & (y_test == 1)).sum())})
display(pd.DataFrame(rows).round(3))

prec, rec, _ = precision_recall_curve(y_test, proba)
px.area(x=rec, y=prec, labels={"x": "Recall", "y": "Precision"},
        title="Precision-Recall curve", height=350).show()

In [ ]:
imp = pd.Series(model.feature_importances_, index=FEATURE_COLUMNS).sort_values().tail(15)
px.bar(imp, orientation="h", title="Top 15 features", height=480,
       labels={"value": "importance", "index": "feature"}).show()
print("Slopes ('getting worse') and days since last failure rank highest, matching the EDA above.")

## 9. Save the model used by the app
`src/train.py` repeats steps 6 to 8 and saves `models/model.joblib`, `metrics.json` and `feature_importance.csv`. The Streamlit dashboard loads these files.

In [ ]:
from src.train import train
saved = train(data_dir=DATA_DIR, model_dir=ROOT / "models")

## Key takeaways
- **EDA showed the warning signs**: vibration and temperature drift up 3 to 10 days before a failure, and current gets erratic.
- **Feature engineering turned those signs into numbers** (rolling stats, slopes) using only past data.
- **Accuracy is misleading** with rare events; judge the model on **recall**, precision and ROC-AUC.
- **Split by time**, never randomly, to mimic real deployment.
- **The threshold is a business decision**: how many false alarms will you accept to avoid a breakdown?